# XGBoost — Nowcast độ mặn tuần tại trạm (ĐBSCL)
chạy trên `weekly_ml_dataset_v2_leftjoin.csv`.

Điểm khác bản cũ (`train_week.ipynb`):
1. **Thước đo = walk-forward theo thời gian** (train 2020–21 / val 2022 / test 2023).
2. **Dùng hết 1960 dòng mặn** (left-join phổ, phổ=NaN khi thiếu ảnh) thay vì 632 dòng inner-join.
3. **Hai model tách biệt**: hồi quy `sal_mean` (dòng `mean_reliable`) + phân loại ngưỡng 1/4 g/L (toàn bộ dòng, `sample_weight`).
4. Feature: thêm `lag1/lag2`, mùa vụ cyclic, `completeness_pct`, `source_format`.
5. Bỏ `year` (không ngoại suy được), cap outlier, target `mean` thay `max`.

In [11]:
#%pip install xgboost scikit-learn pandas numpy

In [12]:
import pandas as pd, numpy as np, warnings; warnings.filterwarnings("ignore")
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
                             precision_recall_fscore_support)

# ---- Cấu hình ----
CSV_PATH   = "weekly_ml_dataset_v2_leftjoin.csv"
SALINITY_CAP = 40.0     # cap giá trị phi vật lý
USE_COMPLETENESS = False # toggle ablation: tắt để kiểm tra completeness_pct có phải proxy protocol đo không
def rmse(a,b): return np.sqrt(mean_squared_error(a,b))

## 1. Load + làm sạch
Ép comma-decimal tự động, cap outlier độ mặn.

In [13]:
df = pd.read_csv(CSV_PATH, parse_dates=["date"])
KEEP_STR = {"station_id","date","station_name_first","river_name_first","source_format","week_date"}
for c in df.columns:
    if c not in KEEP_STR and not pd.api.types.is_numeric_dtype(df[c]):
        cc = pd.to_numeric(df[c].astype(str).str.replace(",",".",regex=False), errors="coerce")
        if cc.notna().mean() > 0.8: df[c] = cc
for c in ["salinity_max_max","salinity_max_mean","salinity_max_min"]:
    df[c] = df[c].clip(upper=SALINITY_CAP)
df = df.sort_values(["station_id","date"]).reset_index(drop=True)
print(df.shape, "| có phổ:", int(df["s2_available"].sum()), "| phổ=NaN:", int((df["s2_available"]==0).sum()))

(1960, 97) | có phổ: 632 | phổ=NaN: 1328


## 2. Feature engineering (Bước 5)
- `lag1/lag2`: độ mặn tuần trước, hai tuần trước (proxy quán tính).
- Mùa vụ **cyclic** sin/cos (day-of-year + week-of-year) thay số nguyên.
- `source_format` one-hot (kèm `unknown` cho 375 dòng chưa verify được nguồn).
- `completeness_pct`: feature phụ để model tách ảnh hưởng cách đo.

In [14]:
g = df.groupby("station_id")
df["salinity_lag1"] = g["salinity_max_mean"].shift(1)
df["salinity_lag2"] = g["salinity_max_mean"].shift(2)
doy = df["day_of_year"]
df["doy_sin"]  = np.sin(2*np.pi*doy/365.25); df["doy_cos"]  = np.cos(2*np.pi*doy/365.25)
df["week_sin"] = np.sin(2*np.pi*df["week"]/52); df["week_cos"] = np.cos(2*np.pi*df["week"]/52)
df["source_format"] = df["source_format"].fillna("unknown")
sf = pd.get_dummies(df["source_format"], prefix="sf")
df = pd.concat([df, sf], axis=1)

STATIC   = ["lon_first","lat_first","DEM_first","Distance_to_River_first","Distance_to_Coast_first"]
SEASON   = ["doy_sin","doy_cos","week_sin","week_cos"]
WEATHER  = ["temperature_2m_c_mean","precipitation_mm_sum","rainy_day_sum","potential_evaporation_mm_sum",
            "runoff_mm_sum","runoff_mm_max","wind_speed_10m_ms_mean","wind_speed_10m_ms_max",
            "surface_pressure_hpa_min","soil_moisture_layer1_vol_mean","solar_radiation_mj_m2_sum"]
SPECTRAL = ["MNDWI_median_week","NDWI_median_week","NDVI_median_week","B11_median_week","B12_median_week",
            "Red_SWIR1_median_week","Red_SWIR2_median_week","BGRratio_median_week","NDCI_median_week"]
LAG      = ["salinity_lag1","salinity_lag2"]
QUALITY  = (["completeness_pct"] if USE_COMPLETENESS else []) + list(sf.columns)
FEATURES = [c for c in STATIC+SEASON+WEATHER+SPECTRAL+QUALITY if c in df.columns]
print(len(FEATURES), "features")

32 features


## 3. Walk-forward split theo thời gian (Bước 7)
train 2020–2021 · val 2022 (early stopping) · test 2023 — đồng bộ với LSTM để so sánh công bằng.

In [15]:
tr  = df["year"].isin([2020,2021])
va  = df["year"]==2022
teY = df["year"]==2023
print(f"train={tr.sum()}  val={va.sum()}  test={teY.sum()}")

train=1010  val=467  test=483


## 4. MODEL 1 — Hồi quy `salinity_max_mean` (Bước 9)
Chỉ huấn luyện/đánh giá trên dòng `mean_reliable=True` (đủ ngày đo trong tuần). Target log1p vì lệch phải.

In [16]:
rel = df["mean_reliable"]==True
mtr, mva, mte = tr&rel, va&rel, teY&rel
yr = np.log1p(df["salinity_max_mean"])

regr = XGBRegressor(objective="reg:squarederror", learning_rate=0.03, max_depth=5,
                    n_estimators=3000, early_stopping_rounds=60, subsample=0.8,
                    colsample_bytree=0.8, random_state=42)
regr.fit(df.loc[mtr,FEATURES], yr[mtr],
         eval_set=[(df.loc[mva,FEATURES], yr[mva])], verbose=False)

pred = regr.predict(df.loc[mte,FEATURES]); yt = np.expm1(yr[mte]); yp = np.expm1(pred)
print(f"n_train={mtr.sum()} n_val={mva.sum()} n_test={mte.sum()}")
print(f"R2={r2_score(yr[mte],pred):.3f} | RMSE={rmse(yt,yp):.2f} g/L | MAE={mean_absolute_error(yt,yp):.2f} g/L")

pred_train = regr.predict(df.loc[mtr, FEATURES])

# ===== Validation =====
pred_val = regr.predict(df.loc[mva, FEATURES])

# ===== Test =====
pred_test = regr.predict(df.loc[mte, FEATURES])

# R² trên log scale
train_r2 = r2_score(yr[mtr], pred_train)
val_r2   = r2_score(yr[mva], pred_val)
test_r2  = r2_score(yr[mte], pred_test)

print(f"Train R² = {train_r2:.3f}")
print(f"Val   R² = {val_r2:.3f}")
print(f"Test  R² = {test_r2:.3f}")
print(regr.best_iteration)
print(regr.best_score)
# Chuyển về g/L
y_train = np.expm1(yr[mtr])
y_val   = np.expm1(yr[mva])
y_test  = np.expm1(yr[mte])

pred_train_g = np.expm1(pred_train)
pred_val_g   = np.expm1(pred_val)
pred_test_g  = np.expm1(pred_test)

print("Train RMSE:", np.sqrt(mean_squared_error(y_train, pred_train_g)))
print("Val   RMSE:", np.sqrt(mean_squared_error(y_val, pred_val_g)))
print("Test  RMSE:", np.sqrt(mean_squared_error(y_test, pred_test_g)))

print("Train MAE :", mean_absolute_error(y_train, pred_train_g))
print("Val   MAE :", mean_absolute_error(y_val, pred_val_g))
print("Test  MAE :", mean_absolute_error(y_test, pred_test_g))

n_train=571 n_val=294 n_test=252
R2=0.752 | RMSE=2.07 g/L | MAE=1.63 g/L
Train R² = 0.969
Val   R² = 0.548
Test  R² = 0.752
168
0.5665898400714651
Train RMSE: 1.728251317048203
Val   RMSE: 2.446733675664568
Test  RMSE: 2.067068692703649
Train MAE : 0.9296880175595674
Val   MAE : 1.879591558818625
Test  MAE : 1.6331571283946638


### 4b. Sai số theo `source_format` (Bước 10 — kiểm tra bias theo loại trạm)

In [17]:
for s in sorted(df.loc[mte,"source_format"].unique()):
    idx = mte & (df["source_format"]==s)
    if idx.sum()>=5:
        e = mean_absolute_error(np.expm1(yr[idx]), np.expm1(regr.predict(df.loc[idx,FEATURES])))
        print(f"  {s:<20} n={idx.sum():3d}  MAE={e:.2f} g/L")

  hourly_dense         n=252  MAE=1.63 g/L


### 4c. Feature importance
> **Lưu ý:** nếu `completeness_pct` đứng top, chạy lại với `USE_COMPLETENESS=False` (cell 1) để kiểm tra
> nó là tín hiệu thật hay chỉ là *proxy của protocol đo* (đo dày hơn vào mùa mặn).

In [18]:
imp = (pd.DataFrame({"feature":FEATURES, "importance":regr.feature_importances_})
       .sort_values("importance", ascending=False).head(20).reset_index(drop=True))
imp

,feature,importance
0,sf_hourly_sparse_2020,0.337941
1,sf_hourly_dense,0.261055
2,Distance_to_Coast_first,0.075507
3,wind_speed_10m_ms_mean,0.044482
4,DEM_first,0.032679
5,lat_first,0.031106
6,wind_speed_10m_ms_max,0.030791
7,lon_first,0.023242
8,Distance_to_River_first,0.016986
9,runoff_mm_sum,0.014401


## 5. MODEL 2 — Phân loại vượt ngưỡng 1 & 4 g/L (Bước 9)
Dùng **`salinity_max_max`** (đỉnh tuần) và **toàn bộ dòng** kể cả phổ=NaN và trạm thưa —
với `sample_weight` thấp cho dòng chất lượng kém thay vì loại bỏ. Đây là đầu ra vận hành cho cảnh báo ranh mặn.

In [19]:
w = np.where(df["mean_reliable"]==True, 1.0,
     np.where(df["mean_reliable"]==False, 0.4, 0.5))   # NaN (chưa verify) -> 0.5

for thr in [1.0, 4.0]:
    yb = (df["salinity_max_max"] >= thr).astype(int)
    clf = XGBClassifier(objective="binary:logistic", learning_rate=0.05, max_depth=4,
                        n_estimators=800, subsample=0.8, colsample_bytree=0.8,
                        eval_metric="logloss", random_state=42)
    clf.fit(df.loc[tr|va,FEATURES], yb[tr|va], sample_weight=w[tr|va])
    pb = clf.predict(df.loc[teY,FEATURES])
    p,r,f,_ = precision_recall_fscore_support(yb[teY], pb, average="binary", zero_division=0)
    print(f"≥{thr:g} g/L  (test n={teY.sum()}, dương={int(yb[teY].sum())}):  P={p:.2f}  R={r:.2f}  F1={f:.2f}")

≥1 g/L  (test n=483, dương=384):  P=0.91  R=0.98  F1=0.94
≥4 g/L  (test n=483, dương=277):  P=0.83  R=0.91  F1=0.87
